In [ ]:
import os
os.chdir(os.path.dirname(os.path.abspath('__file__'))) 

In [ ]:
import weaviate
from utils import dense_search, sparse_search, rrf_fusion, rerank, build_context

In [ ]:
client = weaviate.connect_to_local()

In [ ]:
collection = client.collections.get("AlzheimersChunks")

In [ ]:
query = "How does tau pathology contribute to cognitive decline?"

In [ ]:
def hybrid_retrieval(collection, query, limit=5):
    dense = dense_search(collection, query, limit)
    sparse = sparse_search(collection, query, limit)
    fused = rrf_fusion(dense, sparse)
    return fused[:limit]

In [ ]:
results = hybrid_retrieval(collection, query, limit=25)

In [ ]:
for i, result in enumerate(results):
    print("=" * 40)
    print(f"Title: {result.properties['title']}")
    print(f"Text: {result.properties['text']}")

## Adding Cross Encoder

In [ ]:
reranked_results = rerank(query, results, limit=5)

In [ ]:
for i, result in enumerate(reranked_results):
    doc, score = result
    print("=" * 40)
    print(f"Score: {score:.4f}")
    print(f"Title: {doc.properties['title']}")
    print(f"Text: {doc.properties['text']}")

## Generation

In [ ]:
SYSTEM_PROMPT = """
    You are a biomedical research assistant.

    Answer ONLY using the provided evidence.

    If evidence is insufficient,
    say so explicitly.

    Do not fabricate claims.

    Cite document numbers
    when making statements.

    Highlight conflicting findings
    when present.
"""

In [ ]:
context = build_context(reranked_results)

In [ ]:
def create_prompt(query, context):

    return f"""
        {SYSTEM_PROMPT}

        QUESTION:
        {query}

        EVIDENCE:
        {context}

        ANSWER:
        """

In [ ]:
prompt = create_prompt(query, context)
print(prompt)

In [ ]:
output = create_prompt(query, context)

with open("output.txt", "w", encoding="utf-8") as f:
    f.write(output)

In [ ]:
prompt = create_prompt(query, context)

In [ ]:
import ollama

# Define your prompt
prompt = "How does tau pathology contribute to cognitive decline"

# Call the local MedCPT model
response = ollama.generate(
    model="medgemma1.5:4b-it-q4_K_M",
    prompt=prompt,
    options={
        "stream": False  # Disables streaming for a single response
    }
)

# Print the text output
print(response["response"])